# Activity: Singular Value Decomposition (SVD) of the Stoichiomatric Matrix for Genome-Scale Metabolic Models
In this activity, we will perform Singular Value Decomposition (SVD) on the stoichiometric matrix of a genome-scale metabolic model (GEM) and analyze the results to gain insights into the metabolic capabilities of the organism. Thus, we continue our analysis from the previous module of the covariance matrix computed from the stiochiometric matrix.

> __Learning Objectives:__
> 
> By the end of this activity, you will be able to:
>
> Three learning objectives here.

Let's get started!
___

## Background: What is a stoichiometric matrix?
Suppose we have a set of chemical (or biochemical) reactions $\mathcal{R}$ involving the chemical species (metabolite) set $\mathcal{M}$. Then, the stoichiometric matrix is a $\mathbf{S}\in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}$ matrix that holds the stoichiometric coefficients $\sigma_{ij}\in\mathbf{S}$ such that:
* $\sigma_{ij}>0$: Chemical species (metabolite) $i$ is _produced_ by reaction $j$. Species $i$ is a product of reaction $j$.
* $\sigma_{ij} = 0$: Chemical species (metabolite) $i$ is not connected with reaction $j$
* $\sigma_{ij}<0$: Chemical species (metabolite) $i$ is _consumed_ by reaction $j$. Species $i$ is a reactant of reaction $j$.

Thus, the stoichiometric matrix $\mathbf{S}$ encodes the complete connectivity information of the chemical reaction system for, in this case, a biochemical reaction network. Thus, it is the digital representation of the reaction network inside of a cell.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
We developed a simple software development kit (SDK) against [the BiGG Models application programming interface at the University of California, San Diego](http://bigg.ucsd.edu/). The [BiGG Models database](http://bigg.ucsd.edu/) integrates published genome-scale metabolic networks into a single database with standardized nomenclature and structure. 

> __What are we doing here?__
> 
> We are going to download a stoichiometric matrix from [the BiGG models database](http://bigg.ucsd.edu/) using [the BiGG models API](http://bigg.ucsd.edu/data_access) and then compute its eigendecomposition. 
> * [The BiGG models API](http://bigg.ucsd.edu/data_access) allows users to programmatically access genome-scale stoichiometric model reconstructions using a simple web API. There are `108` models of intracellular biochemistry occurring in various organisms (including humans) in the database (so far); [see here for a list of models](http://bigg.ucsd.edu/models).
> * Here, we'll first explore the [model of Platelet metabolism developed by Palsson and coworkers](https://pubmed.ncbi.nlm.nih.gov/24473230/), which is a curated, functionally tested, and experimentally validated biochemical reaction network of Human platelet metabolism. This model has 738 metabolites and 1008 reactions. 
> 
>  We call the model download endpoint of [the BiGG models API](http://bigg.ucsd.edu/data_access) and then save the model file to disk (so we don't hit the API unless we have to). 

This call returns model information organized as [a Julia dictionary](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) in the `model::Dict{String, Any}` variable. If a model file is saved, we use the cached file instead of making an API call.

In [2]:
model = let

    # build download endpoint -
    baseurl = "http://bigg.ucsd.edu"; # base url to download model
    modelid = "iAT_PLT_636"; # model id to download (change as needed)
    path_to_saved_model_file = joinpath(_PATH_TO_DATA, "saved-model-$(modelid).jld2");

    # check: do we have a model file saved?
    model = nothing;
    if (isfile(path_to_saved_model_file) == false)
        
        endpoint = MyBiggModelsDownloadModelEndpointModel();
        endpoint.bigg_id = modelid;
        url = build(baseurl, endpoint)
        model = MyBiggModelsDownloadModelEndpointModel(url);

        # Before we move on, save this model for later (so we don't keep hitting the API)
        save(path_to_saved_model_file, Dict("model" => model));
    else
        model = load(path_to_saved_model_file)["model"];
    end
    model; # return the model (either saved, or downloaded)
end

JSON.Object{String, Any} with 6 entries:
  "metabolites"  => Any[Object{String, Any}("id"=>"pa_hs_18_2_20_4_c", "name"=>…
  "reactions"    => Any[Object{String, Any}("id"=>"PI4P5K_18_0_20_4", "name"=>"…
  "genes"        => Any[Object{String, Any}("id"=>"8611", "name"=>"PLPP1", "not…
  "id"           => "iAT_PLT_636"
  "compartments" => Object{String, Any}("c"=>"cytosol", "e"=>"extracellular spa…
  "version"      => "1"

__Metabolite records__: Each metabolite (chemical compound) in the network has an associated metabolite record with several fields. Let's take a look at the metabolite at index `1`. The key field for today in the metabolite record is the `id` field, an abbreviation or symbol associated with this metabolite.

In [3]:
model["metabolites"][1] # example metabolite record

JSON.Object{String, Any} with 5 entries:
  "id"          => "pa_hs_18_2_20_4_c"
  "name"        => "Pa hs 18 2 20 4[c]"
  "compartment" => "c"
  "notes"       => Object{String, Any}("original_bigg_ids"=>Any["pa_hs_18_2_20_…
  "annotation"  => Object{String, Any}("bigg.metabolite"=>Any["pa_hs_18_2_20_4"…

__Reaction records__: Similarly, each reaction in the network has a reaction record with several fields. Let's look at the reaction record at index `25`. The key field for the reaction record is the `metabolites` field, which lists the stoichiometric coefficients associated with this particular reaction.

In [4]:
model["reactions"][25] # example reaction record

JSON.Object{String, Any} with 9 entries:
  "id"                 => "PI4P5K_18_1_18_2"
  "name"               => "PI4P5K 18 1 18 2"
  "metabolites"        => Object{String, Any}("adp_c"=>1.0, "atp_c"=>-1.0, "h_c…
  "lower_bound"        => 0.0
  "upper_bound"        => 1000.0
  "gene_reaction_rule" => "200576 or 23396 or 8394 or 8395 or 8396 or 5305 or 7…
  "subsystem"          => "Expanded Glycerophospholipid metabolism"
  "notes"              => Object{String, Any}("original_bigg_ids"=>Any["PI4P5K_…
  "annotation"         => Object{String, Any}("bigg.reaction"=>Any["PI4P5K_18_1…

For each reaction record, we can see the chemical species (metabolites) involved in the reaction and their associated stoichiometric coefficients. Negative coefficients indicate reactants (consumed), while positive coefficients indicate products (produced).

In [5]:
model["reactions"][25]["metabolites"]

JSON.Object{String, Any} with 5 entries:
  "adp_c"                   => 1.0
  "atp_c"                   => -1.0
  "h_c"                     => 1.0
  "pail345p_hs_18_1_18_2_c" => 1.0
  "pail45p_hs_18_1_18_2_c"  => -1.0

__Stoichiometric matrix__: Next, let's build a stoichiometric matrix $\mathbf{S}$ using the metabolite and reaction records. We'll do this using two for loops. 

> __Strategy__: In the outer loop, we iterate over the system's metabolites (chemical species) and select the `id` from the metabolites record for each metabolite. In the inner loop, we iterate over each reaction. For each reaction record, we ask if this reaction has an entry for the current metabolite `id` value; if it does, we grab the stoichiometric coefficient $\sigma_{ij}$ corresponding to this metabolite and reaction.

We'll save the stoichiometric matrix in the `S::Matrix{Float64}` variable.

In [6]:
S = let

    # get some data from the model -
    m = model["metabolites"]; # get list of metabolites
    r = model["reactions"]; # get list of reactions
    number_of_rows = length(m); # how many metabolites do we have? (rows)
    number_of_cols = length(r); # how many reactions do we have? (cols)
    S = zeros(number_of_rows,number_of_cols); # initialize an empty stoichiometric matrix

    # let's build a stm -
    for i ∈ eachindex(m)
        metabolite = m[i]["id"]; # we are checking if this metabolite is in the reaction record
        for j ∈ eachindex(r)
            reaction = r[j];
            if (haskey(reaction["metabolites"], metabolite) == true)
                S[i,j] = reaction["metabolites"][metabolite];
            end
        end
    end
    S; 
end;

___

## Summary
One concise summry sentence goes here.

> __Key Takeaways:__
>
> Three key takeaways go here.

One concise concluding sentence goes here.
___